# 03 - Feature Engineering: TF-IDF & Negation Handling
## Vectorizing Text While Preserving Sentiment Polarity

In this notebook, we explore the feature engineering strategy:
1. **The Stopwords Dilemma**: Why default English stopword removal severely harms sentiment classification.
2. **N-gram Range Exploration**: Comparing unigrams $(1, 1)$, bigrams $(1, 2)$, and trigrams $(1, 3)$.
3. **Sublinear TF Scaling**: Dampening term frequency impacts ($1 + \log(	ext{tf})$).
4. **Vocabulary Extraction**: Inspecting the most predictive positive and negative n-gram tokens.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

df = pd.read_csv("../data/processed/train_reviews_clean.csv")
X = df["clean_review"].fillna("")
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training samples: {len(X_train)} | Test samples: {len(X_test)}")


### 1. The Negation Preservation Strategy
Scikit-learn's default English stopwords list contains **318 words**, including:
`'not', 'no', 'never', 'neither', 'nor', 'hardly', 'barely', 'without'`.
When these are removed:
- *"not good"* $	o$ *"good"*
- *"never recommend"* $	o$ *"recommend"*
- *"waste of time"* $	o$ *"waste time"* (good, but losing negations is disastrous).

We construct a **negation-preserving stopword list** by removing negation words from the stop set:


In [ ]:
NEGATION_WORDS = {
    "not", "no", "never", "neither", "nor", "none", "nobody", "nowhere",
    "hardly", "scarcely", "barely", "cannot", "without", "against"
}

CUSTOM_STOP_WORDS = list(ENGLISH_STOP_WORDS - NEGATION_WORDS)
print(f"Default stop words: {len(ENGLISH_STOP_WORDS)}")
print(f"Negation words preserved: {len(NEGATION_WORDS)}")
print(f"Filtered stop words: {len(CUSTOM_STOP_WORDS)}")


### 2. Comparing Feature Extraction Configurations


In [ ]:
configs = {
    "Standard Unigrams (Default Stopwords)": TfidfVectorizer(max_features=5000, stop_words="english"),
    "Unigrams (Negation Preserved)": TfidfVectorizer(max_features=5000, stop_words=CUSTOM_STOP_WORDS),
    "Bigrams (1, 2) + Sublinear TF": TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.85, sublinear_tf=True, stop_words=CUSTOM_STOP_WORDS),
    "Trigrams (1, 3) + Sublinear TF": TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.85, sublinear_tf=True, stop_words=CUSTOM_STOP_WORDS)
}

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

results = {}
for name, vec in configs.items():
    print(f"Fitting {name}...")
    X_tr = vec.fit_transform(X_train)
    X_te = vec.transform(X_test)
    
    clf = LogisticRegression(C=2.0, max_iter=1000, random_state=42)
    clf.fit(X_tr, y_train)
    acc = accuracy_score(y_test, clf.predict(X_te))
    vocab_size = X_tr.shape[1]
    results[name] = {"Accuracy": acc, "Vocabulary Size": vocab_size}
    print(f"  -> Vocab: {vocab_size:,} | Accuracy: {acc*100:.2f}%")

res_df = pd.DataFrame(results).T
res_df


### 3. Inspecting Top Informative Features


In [ ]:
best_vec = configs["Bigrams (1, 2) + Sublinear TF"]
best_vec.fit(X_train)
X_train_vec = best_vec.transform(X_train)

clf = LogisticRegression(C=2.0, max_iter=1000, random_state=42)
clf.fit(X_train_vec, y_train)

feature_names = np.array(best_vec.get_feature_names_out())
coefs = clf.coef_[0]

# Top 20 positive and negative tokens
top_pos_idx = np.argsort(coefs)[-20:]
top_neg_idx = np.argsort(coefs)[:20]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(feature_names[top_neg_idx], coefs[top_neg_idx], color="#e74c3c")
axes[0].set_title("Top 20 Negative Predictive Features", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Logistic Regression Coefficient")

axes[1].barh(feature_names[top_pos_idx], coefs[top_pos_idx], color="#2ecc71")
axes[1].set_title("Top 20 Positive Predictive Features", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Logistic Regression Coefficient")

plt.tight_layout()
plt.show()


### 4. Summary & Decision:
* Preserving negation tokens and introducing bi-grams drastically improves the representation of key negative phrases (`"waste time"`, `"not good"`, `"worst"`, `"not worth"`).
* **Bigrams with sublinear TF scaling** yields the highest accuracy while maintaining a balanced feature space.
